# Week 9

In [15]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [16]:
predict_conversion_df = pd.read_csv("digital_marketing_campaign_dataset.csv")
google_ads_df = pd.read_csv("GoogleAds_DataAnalytics_Sales_Uncleaned.csv")
marketing_product_df = pd.read_csv("marketing_and_product_performance.csv")

In [17]:
# 1. Clean Numeric Strings (Cost, Sale_Amount)
for col in ['Cost', 'Sale_Amount']:
    google_ads_df[col] = google_ads_df[col].astype(str).str.replace(r'[\$,]', '', regex=True)
    google_ads_df[col] = pd.to_numeric(google_ads_df[col], errors='coerce')

# 2. Impute missing values for primary numeric features with the mean
columns_to_impute = ['Cost', 'Sale_Amount', 'Clicks', 'Impressions', 'Leads']
google_ads_df[columns_to_impute] = google_ads_df[columns_to_impute].fillna(google_ads_df[columns_to_impute].mean())

# 3. Standardize Categorical Columns (Unify typos and case)
# Unify Campaign_Name
google_ads_df['Campaign_Name'] = 'Data Analytics Course' 

# Unify Location to 'hyderabad'
google_ads_df['Location'] = 'hyderabad'

# Lowercase Device and Keyword
google_ads_df['Device'] = google_ads_df['Device'].str.lower()
google_ads_df['Keyword'] = google_ads_df['Keyword'].str.lower()

# 4. Standardize and Parse Dates
# Handling multiple formats: YYYY-MM-DD, DD-MM-YYYY, YYYY/MM/DD
google_ads_df['Ad_Date'] = pd.to_datetime(google_ads_df['Ad_Date'], format='mixed', dayfirst=True)

# Extract Date Features
google_ads_df['DayOfWeek'] = google_ads_df['Ad_Date'].dt.dayofweek
google_ads_df['DayOfMonth'] = google_ads_df['Ad_Date'].dt.day

# 5. Recalculate 'Conversion Rate' to handle missing values and maintain logical consistency
google_ads_df['Conversion Rate'] = google_ads_df['Conversions'] / google_ads_df['Clicks']
google_ads_df['Conversion Rate'] = google_ads_df['Conversion Rate'].replace([np.inf, -np.inf], np.nan).fillna(0)

# Check remaining missing values
print("Remaining missing values in google_ads_df:\n", google_ads_df.isnull().sum())
print("\nCleaned Categorical Samples:")
print(google_ads_df[['Device', 'Ad_Date', 'DayOfWeek']].head())

Remaining missing values in google_ads_df:
 Ad_ID               0
Campaign_Name       0
Clicks              0
Impressions         0
Cost                0
Leads               0
Conversions        74
Conversion Rate     0
Sale_Amount         0
Ad_Date             0
Location            0
Device              0
Keyword             0
DayOfWeek           0
DayOfMonth          0
dtype: int64

Cleaned Categorical Samples:
    Device    Ad_Date  DayOfWeek
0  desktop 2024-11-16          5
1   mobile 2024-11-20          2
2  desktop 2024-11-16          5
3   tablet 2024-11-26          1
4  desktop 2024-11-22          4


C:\Users\santo\AppData\Local\Temp\ipykernel_11376\1740704996.py:3: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  google_ads_df[col] = google_ads_df[col].astype(str).str.replace(r'[\$,]', '', regex=True)
C:\Users\santo\AppData\Local\Temp\ipyk

My approach to each of these datasets was very similar as I aimed to keep a low learning rate and shallow trees. I penalized complexity using min_impurity_decrease, set the minimum samples per leaf to 10, and utilized only 80% of the data for each tree to improve generalization.

### Gradient Boost of predict_conversion_df

In [18]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# 1. Prepare Features and Target
X = predict_conversion_df.drop(columns=['CustomerID', 'Conversion', 'ConversionRate'], errors='ignore')
y = predict_conversion_df['Conversion']

# 2. Encode Categorical Variables
X = pd.get_dummies(X, drop_first=True)

# 3. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Initialize and train Gradient Boosting Classifier with specified constraints
gb_model = GradientBoostingClassifier(
    n_estimators=500,     # Increased estimators to compensate for low learning rate
    learning_rate=0.05,    # Low learning rate
    max_depth=3,           # Shallow tree depth
    min_impurity_decrease=0.01, # Penalize complexity
    min_samples_leaf=10,   # Minimum samples per leaf
    subsample=0.8,         # Use a subsample (Stochastic Gradient Boosting)
    random_state=42
)

gb_model.fit(X_train, y_train)

# 5. Evaluate
y_pred = gb_model.predict(X_test)
print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy Score: 0.9087

Classification Report:
              precision    recall  f1-score   support

           0       0.76      0.36      0.49       194
           1       0.92      0.98      0.95      1406

    accuracy                           0.91      1600
   macro avg       0.84      0.67      0.72      1600
weighted avg       0.90      0.91      0.89      1600



The accuracy score seems high however, there is significant class imbalance in the classification report. 87.87% of the samples were in Class 1 of the support column, meaning that there in the data there are significantly more people that have converted than those who have not. There was poor minority class performance, the model only successfully identified 36% of the people who did not convert, likely mislabeling the missing 64% of those that did not convert. The F1-score was great for Class 1, however this was expected since the model is biased toward this class.

### Gradient Boost for google_ads_df

In [19]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score

# 1. Clean Data (Drop rows where target 'Conversions' is missing)
google_ads_clean = google_ads_df.dropna(subset=['Conversions']).copy()

# 2. Prepare Features and Target
# Drop non-predictive columns and target-related columns
X_ga = google_ads_clean.drop(columns=['Ad_ID', 'Ad_Date', 'Conversions', 'Conversion Rate'], errors='ignore')
y_ga = google_ads_clean['Conversions']

# 3. Encode Categorical Variables (Campaign_Name, Location, Device, Keyword)
X_ga = pd.get_dummies(X_ga, drop_first=True)

# 4. Train-Test Split
X_train_ga, X_test_ga, y_train_ga, y_test_ga = train_test_split(X_ga, y_ga, test_size=0.2, random_state=42)

# 5. Initialize and train Gradient Boosting Regressor with specified constraints
gb_reg_ga = GradientBoostingRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=3,
    min_impurity_decrease=0.01,
    min_samples_leaf=10,
    subsample=0.8,
    random_state=42
)

gb_reg_ga.fit(X_train_ga, y_train_ga)

# 6. Evaluate
y_pred_ga = gb_reg_ga.predict(X_test_ga)
mse_ga = mean_squared_error(y_test_ga, y_pred_ga)
r2_ga = r2_score(y_test_ga, y_pred_ga)

print(f"Gradient Boosting Regressor Results for google_ads_df:")
print(f"Mean Squared Error (MSE): {mse_ga:.4f}")
print(f"R^2 Score: {r2_ga:.4f}")

Gradient Boosting Regressor Results for google_ads_df:
Mean Squared Error (MSE): 5.5716
R^2 Score: -0.0952


Currently, this regressor is performing poorly per the negative R^2 score and the MSE of 5.57. The negative R^2 score means that the model is performing worse than average. Using the RMSE of 2.36, this means that the models prediction is on average off by about 2.4 conversions. There are many possibilities for this. By standardizing certain columns, they became constants which provide no new information and cause the model to have fewer features to work with. It could be that using one-hot encoding I created a sparse dataset with many 0s for columns. 

### Gradient Boost of marketing_product_df

In [20]:
# 1. Prepare Features and Target
# Drop high-cardinality IDs and target-related columns (leaks)
features_to_drop = [
    'Campaign_ID', 'Product_ID', 'Customer_ID', 'Flash_Sale_ID', 
    'Bundle_ID', 'Conversions', 'Revenue_Generated', 'ROI'
]
X_mp = marketing_product_df.drop(columns=features_to_drop, errors='ignore')
y_mp = marketing_product_df['Conversions']

# 2. Encode Categorical Variables
X_mp = pd.get_dummies(X_mp, drop_first=True)

# 3. Train-Test Split
X_train_mp, X_test_mp, y_train_mp, y_test_mp = train_test_split(X_mp, y_mp, test_size=0.2, random_state=42)

# 4. Initialize and train Gradient Boosting Regressor with specified constraints
gb_reg_mp = GradientBoostingRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=3,
    min_impurity_decrease=0.01,
    min_samples_leaf=10,
    subsample=0.8,
    random_state=42
)

gb_reg_mp.fit(X_train_mp, y_train_mp)

# 5. Evaluate
y_pred_mp = gb_reg_mp.predict(X_test_mp)
mse_mp = mean_squared_error(y_test_mp, y_pred_mp)
r2_mp = r2_score(y_test_mp, y_pred_mp)

print(f"Gradient Boosting Regressor Results for marketing_product_df:")
print(f"Mean Squared Error (MSE): {mse_mp:.4f}")
print(f"R^2 Score: {r2_mp:.4f}")

Gradient Boosting Regressor Results for marketing_product_df:
Mean Squared Error (MSE): 86608.7158
R^2 Score: -0.0230


Similarly to the previous dataset, this one shows a negative R^2 score (-0.0230), indicating that he model is struggling to outperform a simple prediction. The MSE is 86608.7, corresponding to an RMSE of about 294.3 conversions. Given the range of conversions in the data, this error is very  large. It may be due to the model's shallow depth or due to a high degree of noise in the data.